# 🛡️ AEGIS — Enterprise RAG System
## Google Colab Walkthrough
**LangChain + LangGraph + TF-IDF Hybrid Retrieval + Evaluation**

---
### Pipeline Flow
```
User Query
    │
    ▼
┌─────────────────────────────────────────────────────────────────┐
│  INGESTION (run once per document)                              │
│  Document → Markdown chunking → Metadata extraction            │
│          → Embedding (text-embedding-3-large) → Pinecone upsert│
└─────────────────────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────────────────────┐
│  RETRIEVAL GRAPH (LangGraph StateGraph)                         │
│  1. Router    → keyword match (fast) or LLM classify (fallback) │
│  2. Expand    → 3 query variants via LLM (multi-query)          │
│  3. HyDE      → hypothetical policy clause embedding            │
│  4. Retrieve  → Dense (Pinecone) + Sparse (TF-IDF) → RRF fusion │
│  5. Filter    → drop stale policy versions                      │
│  6. Rerank    → CrossEncoder ms-marco + TF-IDF calibration      │
│  7. Generate  → create_stuff_documents_chain → grounded answer  │
└─────────────────────────────────────────────────────────────────┘
    │
    ▼
Answer + Full Audit Trail (System/Human/AI/ToolMessage)
```

**Key lecture concepts implemented:**
- Two pipelines (upsert + retrieval)
- Pre-filter + post-filter metadata
- Multi-query expansion + HyDE
- Cross-encoder reranking
- TF-IDF hybrid for precision
- Token limit enforcement
- Caching (in-process / Redis)
- A/B testing harness
- Evaluation metrics (EM, F1, BLEU, Context Precision/Recall)
- LangChain tool calling execution loop
- Pydantic enforcement at every step

## Step 1 — Install Dependencies

In [ ]:
# Install all required packages
# Note: torch is large (~2GB) — use GPU runtime for speed
!pip install -q \
    langgraph>=0.2.0 \
    langchain>=0.3.0 \
    langchain-core>=0.3.0 \
    langchain-community>=0.3.0 \
    langchain-text-splitters>=0.3.0 \
    langchain-openai>=0.2.0 \
    langchain-pinecone>=0.2.0 \
    openai>=1.30.1 \
    pinecone>=3.0.0 \
    pydantic>=2.7.1 \
    tiktoken>=0.7.0 \
    scikit-learn>=1.4.0 \
    sentence-transformers>=2.7.0 \
    torch>=2.2.2 \
    redis

print('✅ All packages installed')

## Step 2 — Set API Keys

In [ ]:
import os
from google.colab import userdata

# Set via Colab Secrets (left sidebar → 🔑 Secrets)
os.environ['OPENAI_API_KEY']  = userdata.get('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
# Optional: os.environ['REDIS_URL'] = userdata.get('REDIS_URL')

print('✅ API keys loaded')
print(f'   OpenAI key prefix: {os.environ["OPENAI_API_KEY"][:8]}...')

## Step 3 — Upload AEGIS source files
Upload the files from the zip: `graph_state.py`, `graph.py`, `retrieval_nodes.py`,
`ingestion.py`, `tfidf_index.py`, `cache.py`, `evaluation.py`, `tools.py`, `utils.py`

In [ ]:
from google.colab import files
uploaded = files.upload()   # select all .py files from the zip
print(f'Uploaded: {list(uploaded.keys())}')

## Step 4 — Verify Graph State Schema (Pydantic)

In [ ]:
# ── Demonstrates Pydantic enforcement at each step ──────────────────────
# Lecture: 'Pydantic for schema enforcement and validation'

from graph_state import AegisState, ChunkResult, VALID_CATEGORIES
import pydantic

print('Valid categories:', sorted(VALID_CATEGORIES))

# Valid state
state = AegisState(
    query='What is the hotel limit for domestic travel?',
    top_k=25,
    num_queries=4,
    use_reranking=True,
    tfidf_calibration_alpha=0.15,
)
print(f'\n✅ Valid state created:')
print(f'   query={state.query}')
print(f'   top_k={state.top_k}, num_queries={state.num_queries}')
print(f'   tfidf_alpha={state.tfidf_calibration_alpha}')

# Invalid alpha → ValidationError
try:
    AegisState(query='test', tfidf_calibration_alpha=2.0)
    print('ERROR: should have raised')
except pydantic.ValidationError as e:
    print(f'\n✅ Pydantic correctly rejected alpha=2.0: {e.errors()[0]["msg"]}')

# Invalid category coerced to General
c = ChunkResult(chunk_id='x', document_id='d1', policy_category='Unknown',
                chunk_text='test text')
print(f'\n✅ Invalid category coerced to: {c.policy_category}')

## Step 5 — TF-IDF Hybrid Retrieval Demo (no API needed)

In [ ]:
# ── TF-IDF index demonstration ───────────────────────────────────────────
# Lecture: 'Re-ranking: retrieve many candidates first, then reorder'
# Our addition: TF-IDF precision calibration via RRF

from tfidf_index import TfidfIndex, reciprocal_rank_fusion, tfidf_calibrate_scores
from graph_state import ChunkResult

# Simulate the policy corpus
corpus = [
    '[Travel Policy] Employees must submit expense reports within 30 days.',
    '[Travel Policy] The per diem rate for international travel is $75 per day.',
    '[Travel Policy] Taxi and rideshare expenses are reimbursable up to $50 per trip.',
    '[Travel Policy] Hotel accommodation is capped at $200 per night for domestic.',
    '[HR Policy] Maternity leave is 16 weeks at full pay.',
    '[HR Policy] Leave requests must be submitted 14 days in advance.',
    '[Travel Policy] GCTEM manages all corporate travel vendor relationships.',
    '[Travel Policy] All flights must be economy class unless journey exceeds 6 hours.',
]

idx = TfidfIndex(corpus)
print(f'TF-IDF index built on {len(idx)} chunks. Ready: {idx.is_ready}')

# Query 1: exact number match
q1  = 'What is the per diem rate international travel $75'
r1  = idx.query(q1, top_k=3)
print(f'\n🔍 Query: "{q1}"')
for r in r1:
    print(f'   [{r.tfidf_score:.4f}] {corpus[r.chunk_index][:70]}')

# Query 2: 30 day deadline
q2 = '30 day expense submission deadline'
r2 = idx.query(q2, top_k=3)
print(f'\n🔍 Query: "{q2}"')
for r in r2:
    print(f'   [{r.tfidf_score:.4f}] {corpus[r.chunk_index][:70]}')

# Demonstrate calibration: TF-IDF bonus overrides small CE gap
chunks_cal = [
    ChunkResult(chunk_id='A', document_id='d1', policy_category='Travel',
                chunk_text='Per diem rate $75 international travel 30 days',
                rerank_score=0.80),
    ChunkResult(chunk_id='B', document_id='d2', policy_category='HR',
                chunk_text='General company culture and values statement',
                rerank_score=0.82),
]
calibrated = tfidf_calibrate_scores('per diem $75 international', chunks_cal, alpha=0.15)
print(f'\n✅ After TF-IDF calibration (alpha=0.15):')
for c in calibrated:
    print(f'   {c.chunk_id}: calibrated_score={c.rerank_score:.4f}')
print(f'   Winner: {calibrated[0].chunk_id} (exact term match promoted despite lower CE score)')

## Step 6 — Message Protocol Demo (System/Human/AI/Tool)

In [ ]:
# ── Message type demonstration ──────────────────────────────────────────
# Lecture: 'Execution loop: LLM → tool call → tool output appended to history'

from graph_state import SystemMessage, HumanMessage, AIMessage, ToolMessage, tool_log
import json

# Simulate one pipeline's message history
messages = [
    SystemMessage(content='AEGIS RAG Pipeline — all decisions logged as ToolMessages.'),
    HumanMessage(content='What is the maximum hotel rate for domestic travel?'),
    tool_log(
        tool_name='router',
        reason="Keyword 'hotel' matched → category=Travel (confidence=high, 0 LLM calls)",
        inputs={'query': 'What is the maximum hotel rate for domestic travel?', 'matched_keyword': 'hotel'},
        outputs={'category': 'Travel', 'confidence': 'high'},
    ),
    AIMessage(content='Policy on rideshare and Uber\nGround transportation reimbursement\nCab fare corporate travel'),
    tool_log(
        tool_name='retrieve_tfidf',
        reason='TF-IDF scored 18 dense candidates against raw query.',
        inputs={'corpus_size': 18, 'query': 'hotel domestic travel'},
        outputs={'top_tfidf_chunk_ids': {'TRV-POL-1001-V4_3': 0.4821, 'TRV-POL-1001-V4_7': 0.3102}},
    ),
    AIMessage(content='The maximum hotel accommodation rate for domestic travel is **$200 per night**, '
              'as specified in Section 7 of the Corporate Travel Policy (TRV-POL-1001-V4).'),
]

print('📋 Full message audit trail:')
print('=' * 60)
for msg in messages:
    role = type(msg).__name__
    if isinstance(msg, ToolMessage):
        data = json.loads(msg.content)
        print(f'[{role}] tool={data["tool"]}')
        print(f'        reason: {data["reason"][:80]}')
        print(f'        outputs: {str(data["outputs"])[:80]}')
    else:
        print(f'[{role}] {msg.content[:100]}')
    print()

print(f'Total messages in audit trail: {len(messages)}')
print(f'Message types: { {type(m).__name__ for m in messages} }')

## Step 7 — Full Pipeline Run (requires API keys)

In [ ]:
# ── Full pipeline run ───────────────────────────────────────────────────
# Lecture: 'Two pipelines: upsert and retrieval'
# Make sure documents are ingested first (see Step 8 for ingestion)

from graph import run_query
import time

query = 'What is the maximum hotel cost per night for domestic travel?'
print(f'🔍 Query: {query}\n')

t0     = time.time()
result = run_query(
    query,
    top_k=25,           # retrieve Top-25 candidates
    num_queries=4,      # 4 expansion variants + HyDE
    use_reranking=True, # cross-encoder + TF-IDF calibration
    tfidf_alpha=0.15,   # TF-IDF precision boost weight
    use_cache=True,     # cache result for repeated queries
)
elapsed = round(time.time() - t0, 2)

print(f'✅ Answer ({elapsed}s, cache_hit={result["cache_hit"]}):')
print(f'   {result["answer"]}')
print(f'\n📊 Pipeline stats:')
print(f'   Category: {result["category"]}')
print(f'   Retrieved (post-filter): {result["retrieved"]}')
print(f'   Sources used: {len(result["sources"])}')
print(f'   RRF applied: {result.get("rrf_applied", False)}')

print(f'\n📎 Top source:')
if result['sources']:
    s = result['sources'][0]
    print(f'   doc={s["document_id"]}  score={s["rerank_score"]:.4f}')
    print(f'   section={s["h1_header"]} / {s["h2_header"]}')
    print(f'   text: {s["chunk_text"][:200]}...')

## Step 8 — Document Ingestion

In [ ]:
# ── Document ingestion ──────────────────────────────────────────────────
# Lecture: 'Chunking and metadata matter most during upsert'

from ingestion import ingest_document
from utils import clean_text

# Upload a .txt policy document
from google.colab import files
uploaded_docs = files.upload()  # upload e.g. 'Travel Policy.txt'

for filename, content in uploaded_docs.items():
    text   = clean_text(content.decode('utf-8', errors='replace'))
    result = ingest_document(text)

    print(f'\n📄 Ingested: {filename}')
    print(f'   document_id : {result["document_id"]}')
    print(f'   category    : {result["category"]}')
    print(f'   chunks      : {result["chunks"]}')
    print(f'   upserted    : {result["upserted"]}')

    print('\n   Ingestion audit messages:')
    import json
    from graph_state import ToolMessage
    for msg in result.get('messages', []):
        if isinstance(msg, ToolMessage):
            d = json.loads(msg.content)
            print(f'   [TOOL] {d["tool"]} — {d["reason"][:60]}')

## Step 9 — A/B Testing (with/without reranking, varying topK)

In [ ]:
# ── A/B test harness ────────────────────────────────────────────────────
# Lecture: 'A/B test retrieval strategies (with/without re-ranking,
#  varying query expansion and topK)'

from evaluation import ab_test, DEFAULT_TRUTH_SET, run_eval
from graph import run_query

# Config A: full pipeline (reranking + TF-IDF calibration)
def config_a(query):
    return run_query(query, top_k=25, num_queries=4,
                     use_reranking=True, tfidf_alpha=0.15)

# Config B: no reranking — vector score only
def config_b(query):
    return run_query(query, top_k=25, num_queries=4,
                     use_reranking=False, tfidf_alpha=0.0)

# Use first 3 questions from truth set for quick demo
truth_subset = DEFAULT_TRUTH_SET[:3]

report_a, report_b, comparison = ab_test(
    truth_set=truth_subset,
    pipeline_a=config_a,
    pipeline_b=config_b,
    name_a='Full Pipeline (rerank+TF-IDF)',
    name_b='No Reranking (vector only)',
)

print('📊 A/B Test Results')
print('=' * 70)
print(report_a.summary())
print(report_b.summary())
print(f'\n🏆 Overall winner: {comparison["overall_winner"]}')
print('\nPer-metric comparison:')
for metric, vals in comparison['metrics'].items():
    winner = vals['winner']
    delta  = vals['delta_B_minus_A']
    print(f'  {metric:<30} delta={delta:+.4f}  winner={winner}')

## Step 10 — Evaluation Metrics (EM, F1, BLEU, Context Precision/Recall)

In [ ]:
# ── Evaluation metrics demo ──────────────────────────────────────────────
# Lecture: 'RAG evaluation metrics: precision/recall, BLEU, EM-style truth sets'

from evaluation import run_eval, DEFAULT_TRUTH_SET
from graph import run_query

print(f'Truth set has {len(DEFAULT_TRUTH_SET)} questions')
print('Sample questions:')
for e in DEFAULT_TRUTH_SET[:3]:
    print(f'  Q: {e.question}')
    print(f'  A: {e.gold_answer}  (category: {e.category})')
    print()

# Run evaluation on first 5 questions
report = run_eval(
    truth_set=DEFAULT_TRUTH_SET[:5],
    pipeline_fn=lambda q: run_query(q, use_cache=True),
    config_name='AEGIS Full Pipeline',
)

print('\n📈 Evaluation Report')
print('=' * 70)
print(report.summary())

print('\nPer-question breakdown:')
for r in report.results:
    print(f'  Q: {r.question[:50]}')
    print(f'     Gold: {r.gold_answer[:40]}')
    print(f'     Pred: {r.predicted_answer[:60]}')
    print(f'     EM={r.exact_match:.2f}  F1={r.token_f1:.2f}  '
          f'BLEU1={r.bleu1:.2f}  Found={r.answer_found}  {r.latency_s:.1f}s')
    print()

## Step 11 — Tool Calling Execution Loop

In [ ]:
# ── LangChain tool calling execution loop ────────────────────────────────
# Lecture: 'LLM → tool call → tool execution → tool output appended to
#  history → LLM final answer'
# Lecture: 'bind tools, inspect ai_message.tool_calls, invoke the tool'

from tools import run_tool_loop, AEGIS_TOOLS
from graph_state import AIMessage, ToolMessage
import json

print(f'Registered tools: {[t.name for t in AEGIS_TOOLS]}')
print()

query = 'What is the hotel limit and per diem for international travel?'
result = run_tool_loop(query, max_iterations=3)

print(f'✅ Final answer (after {result["iterations"]} iteration(s)):')
print(f'   {result["answer"]}')

print(f'\n📋 Message history ({len(result["messages"])} messages):')
for msg in result['messages']:
    role = type(msg).__name__
    if isinstance(msg, ToolMessage):
        try:
            d = json.loads(msg.content)
            print(f'  [{role}] tool={d.get("tool", "?")} reason={d.get("reason","")[:60]}')
        except:
            print(f'  [{role}] {msg.content[:80]}')
    elif isinstance(msg, AIMessage):
        tc = getattr(msg, 'tool_calls', [])
        if tc:
            print(f'  [{role}] → tool_calls: {[t["name"] for t in tc]}')
        else:
            print(f'  [{role}] {msg.content[:80]}')
    else:
        print(f'  [{type(msg).__name__}] {msg.content[:80]}')

## Step 12 — Cache Demo

In [ ]:
# ── Caching layer demonstration ──────────────────────────────────────────
# Lecture: 'Introduce caching layers (e.g., Redis) to reduce latency
#  and operational costs'

from cache import get_cache, AegisCache
from graph import run_query
import time

cache = get_cache()
print(f'Cache backend: {cache.backend_name}')

query = 'What is the hotel limit for domestic travel?'

# First call — cache miss
t0     = time.time()
result = run_query(query, use_cache=True)
t1     = round(time.time() - t0, 2)
print(f'\n1st call  cache_hit={result["cache_hit"]}  latency={t1}s')

# Second call — cache hit
t0     = time.time()
result = run_query(query, use_cache=True)
t2     = round(time.time() - t0, 4)
print(f'2nd call  cache_hit={result["cache_hit"]}  latency={t2}s')

if t1 > 0:
    print(f'\n🚀 Cache speedup: {t1/max(t2,0.001):.0f}×  '
          f'(saving ~{t1-t2:.1f}s per repeated query)')